In [12]:
import os
from openai import OpenAI
from IPython.display import display, Code, Markdown

# #硅基流动API
# ds_api_key = "sk-atisrejlfxlvnymvfxoesps"
# client = OpenAI(api_key=ds_api_key, 
#                 base_url="https://api.siliconflow.cn/v1")

ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')

In [13]:
# 打开并读取Markdown文件
with open('./data/LC数据字典.md', 'r', encoding='utf-8') as f:
    md_content = f.read()
    
len(md_content)

1606

In [16]:
#基于md_content作为模型背景信息，向模型进行相关提问
response = client.chat.completions.create(
    # model="deepseek-ai/DeepSeek-V2.5", 
    model="deepseek-chat",  
    messages=[
        {"role": "system", "content": md_content}, 
        # "content": '请帮我统计下LC数据表一共有哪些字段？共计多少个？'
        {"role": "user", "content": '请帮我介绍下LC数据表'}
    ],
)
display(Markdown(response.choices[0].message.content))

# LC数据表介绍

LC数据表是拍拍贷互联网金融公司在2015年1月1日至2017年1月30日期间记录的贷款用户信息数据集，包含30余万条贷款记录。

## 数据表特点

1. **时间跨度**：覆盖2015-2017年两年多的贷款数据
2. **数据规模**：包含超过30万条贷款记录
3. **数据质量**：由拍拍贷平台采集，并通过回访确认，准确性和可信度高

## 数据维度

该表包含多维度的用户信息，可分为以下几类：

1. **基本信息**：
   - 年龄
   - 性别

2. **认证信息**：
   - 户口认证状态
   - 征信认证状态

3. **信用信息**：
   - 初始评级(A-F)
   - 历史正常还款期数
   - 历史逾期还款期数
   - 总待还本金

4. **贷款信息**：
   - 借款金额
   - 借款类型(电商/APP闪电/普通/其他)

## 应用价值

该数据集可用于：
- 分析逾期用户的特征模式
- 识别高风险贷款用户
- 优化信用评级模型
- 制定风险控制策略
- 提高业务收入同时降低逾期风险

## 数据完整性

表中包含完整的主键(序号)和各维度字段，无明显缺失值问题，适合进行多维度的数据分析。

In [17]:
def get_sql_result(sql_query):
    """
    查询数据库相关数据的函数
    :param sql_query: 必要参数，字符串类型，用于表示查询数据的sql语句；
    :return：sql_query表示的sql语句查询到的结果;
    """
    connection = pymysql.connect(
            host='localhost',  # 数据库地址
            user='root',  # 数据库用户名
            passwd='boboadmin',  # 数据库密码
            db='testdb',  # 数据库名
            charset='utf8'  # 字符集选择utf8
        )
    
    try:
        with connection.cursor() as cursor:
            # SQL查询语句
            sql = sql_query
            cursor.execute(sql)

            # 获取查询结果
            results = cursor.fetchall()

    finally:
        connection.close()
    
    
    return json.dumps(results)

In [42]:
import inspect
import json
import os
from openai import OpenAI
import pymysql
from IPython.display import display, Code, Markdown

#用于自动生成外部函数描述信息
def auto_function_desc(function): #参数为外部函数对象
    #定义一个内部函数用于生成外部函数的完整描述信息
    def inner(function):
        function_description = inspect.getdoc(function)#外部函数的函数说明
        function_name = function.__name__ #外部函数名
        
        system_prompt = '以下是某的函数说明：%s' % function_description
        
        user_prompt = '根据这个函数的函数说明，请帮我创建一个JSON格式的字典，这个字典有如下5点要求,请你仔细阅读，并且务必遵从所有要求：\
                       1.字典总共有三个键值对；\
                       2.第一个键值对的Key是字符串name，value是该函数的名字：%s，也是字符串；\
                       3.第二个键值对的Key是字符串description，value是该函数的函数的功能说明，也是字符串；\
                       4.第三个键值对的Key是字符串parameters，value是一个JSON Schema对象，用于说明该函数的参数输入规范。\
                       5.输出结果必须是一个JSON格式的字典，并且一定不要任何前后修饰语句,务必参按照如下格式进行输出:%s' % (function_name,'{key:value}')
        
        # api_key = "xxx"
        # client = OpenAI(api_key=ds_api_key, 
        #         base_url="https://api.siliconflow.cn/v1")

        ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
        client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
        response = client.chat.completions.create(
        # model="deepseek-ai/DeepSeek-V2.5",  
        # model="deepseek-chat",  
        model="deepseek-reasoner",  
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}

             ]
        )

        return json.loads(response.choices[0].message.content)
    #由于模型根据提示信息生成的外部函数完整信息可能会有问题，因此，如果出现问题则loads环节会报错，则要求模型重新进行生成
    max_try_count = 5 #模型调用的最大次数
    count = 0 #当前调用模型的次数
    while count < max_try_count:
        try:
            function_desc = inner(function)
            break
        except Exception as e:
            count += 1
            print('something error:',e)
            if count == max_try_count:
                print('模型达到最大尝试次数，程序停止！')
                raise
            else:
                print('模型重新生成中......')
    tools = [
    {
        "type": "function", 
        "function":function_desc
    }
]
    return tools

In [43]:
auto_function_desc(get_sql_result)

[{'type': 'function',
  'function': {'name': 'get_sql_result',
   'description': '查询数据库相关数据的函数',
   'parameters': {'type': 'object',
    'properties': {'sql_query': {'type': 'string',
      'description': '用于表示查询数据的sql语句'}},
    'required': ['sql_query']}}}]

In [45]:
#available_functions表示外部函数库
def auto_run_conversation(messages,available_functions=None):
    # api_key = "sk-atisrrfnrxsnulmuvlzqnvuvcglkriejlfxlvnymvfxoesps"
    # client = OpenAI(api_key=ds_api_key, 
                # base_url="https://api.siliconflow.cn/v1")
    ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
    client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
    
    # 如果没有外部函数库，则执行普通的对话任务
    if available_functions == None:
        print('模型原生能力解决该提问.........')
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V2.5",  
            messages=messages
        )
        final_response = response.choices[0].message.content
    else:
        #外部函数库定义
        available_functions = available_functions
        
       #step_3:外部函数完整描述定义 + #step_4:tools参数值定义
        tools = auto_function_desc(available_functions['function'])

        #step_5:第一次模型调用
        response = client.chat.completions.create(
            # model="deepseek-ai/DeepSeek-V2.5",  
            # model="deepseek-chat",  
            model="deepseek-reasoner",  
            messages=messages,
            tools=tools,
        )
        response_message = response.choices[0].message
        
        #判断返回结果是否存在tool_calls，即判断是否需要调用外部函数来回答问题
        if response_message.tool_calls:
            print('function_calling解决该提问.........')
            sql = response_message.tool_calls[0].function.arguments
            print('生成的sql为：:',sql)
            choose = input('是否执行上述sql? y/n')
            if choose == 'n':
                print("您选择不执行sql语句，再见！")
                return
            #step_6:外部函数手动调用且获取调用结果
            fuction_to_call = available_functions['function'] #函数对象
            function_args = json.loads(response_message.tool_calls[0].function.arguments)#函数参数

            function_response = fuction_to_call(**function_args)#函数手动调用

            #step_7:向messages进行两次消息追加
            messages.append(response_message.model_dump())  
            messages.append({
                        "role": "tool",
                        "content": function_response,
                        "tool_call_id":response_message.tool_calls[0].id
                    })

            #step_8: 再次调用大模型
            second_response = client.chat.completions.create(
                model="deepseek-ai/DeepSeek-V2.5",
                messages=messages)
            final_response = second_response.choices[0].message.content
        else:
            final_response = response_message.content
    return Markdown(final_response)

In [ ]:
#测试

In [46]:
messages = [
    {"role": "system", "content": md_content},
    {"role": "user", "content": "请问LC数据表有多少男性用户？"}
]
#定义外部函数库
available_functions = {
            "function": get_sql_result,
        }
auto_run_conversation(messages,available_functions)

function_calling解决该提问.........
生成的sql为：: {"sql_query":"SELECT COUNT(*) AS 男性用户数量 FROM LC WHERE 性别 = '男'"}


是否执行上述sql? y/n y


RuntimeError: 'cryptography' package is required for sha256_password or caching_sha2_password auth methods